# Player availability at the next pick

- **Question:** Given that a source player is still available immediately after the current pick, what is the probability that the player is selected before, or remains available at, a later overall pick?
- **Data:** The latest cutoff-safe canonical ADP row, its source-reported standard deviation or min/max range, and versioned fallback assumptions only when observed spread is missing.
- **Unit of observation:** One source player in one immutable snapshot paired with a current-pick and next-pick scenario.
- **Target:** The real event `selected before next pick`. Linked historical draft outcomes do not yet exist, so the current output is an empirical distribution estimate, not a trained or calibrated classifier.
- **Feature cutoff:** The ADP capture and every spread input must exist at or before the scenario timestamp; later snapshots and later draft results are forbidden.
- **Validation:** Chronological calibration with linked draft outcomes, Brier score, and reliability curves once enough outcomes exist. Current calibration status is unavailable.
- **Scope boundary:** Availability is separate from player quality and roster fit. This notebook does not recommend a player, rank draft choices, or simulate a draft.

## Read the validated Phase 5 board

If the board is unavailable, run the following from the repository root:

```powershell
fantasy-draft data load-adp
fantasy-draft models build-adp-baselines
fantasy-draft data audit
```

The service below opens the published warehouse read-only and returns no player rows when lineage or count checks fail.

In [ ]:
from __future__ import annotations

import math

import pandas as pd

from fantasy_draft_ai.config import find_project_root, load_config
from fantasy_draft_ai.models.adp.availability import (
    estimate_availability,
    estimate_pick_spread,
)
from fantasy_draft_ai.models.adp.config import load_availability_config
from fantasy_draft_ai.models.adp.movement import AdpIdentity
from fantasy_draft_ai.services.adp_market import load_adp_market_board

PROJECT_ROOT = find_project_root()
app_config = load_config()
availability_rules = load_availability_config(
    PROJECT_ROOT / "configs" / "adp_availability.yaml"
)
market_board = load_adp_market_board(app_config)
print({
    "phase5_available": market_board.available,
    "status": market_board.status.message,
    "latest_market_rows": len(market_board.rows),
})

## Keep the evidence visible

The scale-selection order is source-reported standard deviation, then a scale derived from source min/max, then a labeled configuration fallback. A fallback is an assumption, not measured evidence. Canonical IDs may be null; unresolved source identities remain distinct and are never joined by display name.

In [ ]:
market_evidence = pd.DataFrame(
    [
        {
            "player_name": row.player_name,
            "position": row.position,
            "source": row.source,
            "captured_at": row.captured_at,
            "average_pick": row.average_pick,
            "scale": row.availability_scale,
            "evidence_method": row.availability_evidence_method,
            "fallback_group": row.availability_fallback_group,
            "sample_size": row.sample_size,
            "mapping_confidence": row.mapping_confidence,
        }
        for row in market_board.rows[:15]
    ]
)
market_evidence

## Calculate a conditional next-pick probability

The player is known to be available after the current pick, so the estimate conditions on surviving to that point. The two returned probabilities are complements. This deterministic fixture demonstrates the package API and is not a real-player recommendation.

In [ ]:
fixture_estimate = estimate_availability(
    identity=AdpIdentity(source="fixture", raw_source_row_id="WR_EXAMPLE"),
    position="WR",
    average_pick=36.0,
    current_pick=24.0,
    next_pick=48.0,
    observed_standard_deviation=8.0,
    sample_size=100,
    config=availability_rules,
)
assert math.isclose(
    fixture_estimate.probability_selected_before_next_pick
    + fixture_estimate.probability_available_at_next_pick,
    1.0,
)
pd.DataFrame(
    [
        {
            "average_pick": fixture_estimate.average_pick,
            "current_pick": fixture_estimate.current_pick,
            "next_pick": fixture_estimate.next_pick,
            "standard_deviation": fixture_estimate.standard_deviation,
            "evidence": fixture_estimate.evidence_label,
            "P(selected before next)": (
                fixture_estimate.probability_selected_before_next_pick
            ),
            "P(available at next)": (
                fixture_estimate.probability_available_at_next_pick
            ),
        }
    ]
)

In [ ]:
future_picks = (25, 30, 36, 42, 48, 60)
availability_curve = pd.DataFrame(
    [
        {
            "next_pick": next_pick,
            "probability_available": estimate_availability(
                identity=fixture_estimate.identity,
                position="WR",
                average_pick=36.0,
                current_pick=24.0,
                next_pick=float(next_pick),
                observed_standard_deviation=8.0,
                sample_size=100,
                config=availability_rules,
            ).probability_available_at_next_pick,
        }
        for next_pick in future_picks
    ]
)
assert availability_curve["probability_available"].between(0.0, 1.0).all()
assert (availability_curve["probability_available"].diff().dropna() <= 0.0).all()
availability_curve

## Audit observed evidence versus fallback assumptions

These three deterministic calls show the priority explicitly. The fallback row stays labeled so a consumer cannot confuse a versioned assumption with source-measured variation.

In [ ]:
spread_cases = {
    "source standard deviation": estimate_pick_spread(
        position="WR", average_pick=70.0, observed_standard_deviation=6.0,
        minimum_pick=40.0, maximum_pick=100.0, sample_size=80,
        config=availability_rules,
    ),
    "source min/max": estimate_pick_spread(
        position="WR", average_pick=70.0, observed_standard_deviation=None,
        minimum_pick=40.0, maximum_pick=100.0, sample_size=80,
        config=availability_rules,
    ),
    "configured fallback": estimate_pick_spread(
        position="WR", average_pick=70.0, observed_standard_deviation=None,
        minimum_pick=None, maximum_pick=None, sample_size=None,
        config=availability_rules,
    ),
}
assert [spread.method for spread in spread_cases.values()] == [
    "observed_source_stddev", "min_max_derived", "configured_fallback"
]
pd.DataFrame(
    [
        {
            "case": label,
            "method": spread.method,
            "standard_deviation": spread.standard_deviation,
            "fallback_used": spread.fallback_used,
            "evidence_label": spread.evidence_label,
        }
        for label, spread in spread_cases.items()
    ]
)

## Apply the same API to one published market row

When the Phase 5 board is built, this cell chooses the row nearest ADP 36 only to make the example deterministic. It reports evidence and probabilities; it does not compare player quality or advise a selection.

In [ ]:
published_scenario = pd.DataFrame()
if not market_board.available or not market_board.rows:
    print("No validated market board is available; run the prerequisite commands above.")
else:
    example_row = min(
        market_board.rows,
        key=lambda row: (abs(row.average_pick - 36.0), row.source, row.raw_source_row_id),
    )
    current_pick = max(1.0, float(int(example_row.average_pick) - 12))
    next_pick = current_pick + 24.0
    published_estimate = example_row.estimate_availability(
        current_pick=current_pick,
        next_pick=next_pick,
        config=market_board.availability_config or availability_rules,
    )
    published_scenario = pd.DataFrame(
        [
            {
                "player_name": example_row.player_name,
                "source": example_row.source,
                "captured_at": example_row.captured_at,
                "average_pick": example_row.average_pick,
                "current_pick": current_pick,
                "next_pick": next_pick,
                "spread_method": published_estimate.spread_method,
                "P(selected before next)": (
                    published_estimate.probability_selected_before_next_pick
                ),
                "P(available at next)": (
                    published_estimate.probability_available_at_next_pick
                ),
            }
        ]
    )
published_scenario

In [ ]:
pd.DataFrame(
    [
        ("production snapshots", market_board.status.snapshot_count),
        ("availability rows", market_board.status.availability_rows),
        ("calibration", market_board.status.calibration_status),
        ("supervised model", market_board.status.supervised_status),
    ],
    columns=["capability", "status_or_count"],
)

## Interpretation boundary

The current archive contains one independent production ADP capture and no linked historical draft outcomes. The distribution formula can produce a bounded, monotonic probability, but that probability is uncalibrated. It must not be described as a learned survival model or used as proof that a player will last. Continue archiving dated snapshots and real draft outcomes; evaluate calibration chronologically before availability can support a Phase 6 recommendation or simulation engine.